### Загрузка данных

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
df = pd.read_csv('../../data/raw/dataset.csv', index_col=0)
df.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6


In [4]:
df_features = pd.read_csv('../../data/raw/dataset_features.csv')
df_features.head()

,Регион,Год,Заболеваемость_инфекции_на_1000,Коэффициент_смертности_населения,Численность_населения,Оборотная_вода_млн_м3
0,Белгородская область,2004,805.8,16.2,1511.7,1610.0
1,Брянская область,2004,742.7,19.1,1344.1,63.0
2,Владимирская область,2004,915.9,20.1,1497.6,339.0
3,Воронежская область,2004,543.1,18.5,2364.9,2419.0
4,Ивановская область,2004,849.5,21.6,1116.7,223.0


### Merge df_main + df_features

#### Проверка правильности регионов

In [5]:
def compare_unique(df1, df2):
    shared = set(df1.columns) & set(df2.columns)
    differences = {
        col: {
            'only_in_df1': sorted(set(df1[col].unique()) - set(df2[col].unique())),
            'only_in_df2': sorted(set(df2[col].unique()) - set(df1[col].unique()))
        }
        for col in shared
        if set(df1[col].unique()) != set(df2[col].unique())
    }
    identical = [col for col in shared if set(df1[col].unique()) == set(df2[col].unique())]
    if 'Регион' in identical:
        print("Различий уникальных значений по столбцу 'Регион' нет.")
    elif 'Регион' in differences:
        print("Есть различия по уникальным значениям в столбце 'Регион'.")
    return differences, identical


In [6]:
compare_unique(df, df_features)

Различий уникальных значений по столбцу 'Регион' нет.


({}, ['Год', 'Регион'])

#### Соединяем таблицы по двум ключам (inner join)

In [7]:
df.shape

(1992, 10)

In [8]:
df.shape

(1992, 10)

In [9]:
df = pd.merge(
    df, df_features,
    on=['Регион', 'Год'],
    how='inner',  
    suffixes=('_df1', '_df2')
)


In [10]:
df.rename(columns={
    'Объем_сточных_вод_млн_м3': 'V_сточ_вод_млн_м3',
    'Инвестиции_в_ООС': 'Инв_в_ООС',
    'ВРП': 'ВРП',
    'Заболеваемость_инфекции_на_1000': 'Забол_инф_на_1000',
    'Коэффициент_смертности_населения': 'Коэф_смерт_насел',
    'Численность_населения': 'Числ_насел',
    'Оборотная_вода_млн_м3': 'Оборот_вод_млн_м3'
}, inplace=True)

In [11]:
df.columns

Index(['Регион', 'Год', 'V_сточ_вод_млн_м3', 'Инвестиции_в_ООС_тыс_руб',
       'ВРП_млн_руб', 'ВРП_на_душу_руб', 'Индексы_производства_продукции_СХ_%',
       'Доля_городского_населения_%', 'Использование_свеж_воды_млн_м3',
       'Индекс_промыш_производства_%', 'Забол_инф_на_1000', 'Коэф_смерт_насел',
       'Числ_насел', 'Оборот_вод_млн_м3'],
      dtype='object')

In [12]:
df.head()

,Регион,Год,V_сточ_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Забол_инф_на_1000,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2,NaN,14.3,2641.1,1216.0
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4,NaN,14.7,2621.1,998.0
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1,NaN,15.7,2602.6,983.0
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5,NaN,15.9,2572.0,984.0
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6,977.0,15.9,2539.4,969.0


In [13]:
df.shape

(1992, 14)

In [14]:
df.isna().sum()

Регион                                   0
Год                                      0
V_сточ_вод_млн_м3                        5
Инвестиции_в_ООС_тыс_руб                 5
ВРП_млн_руб                              5
ВРП_на_душу_руб                         36
Индексы_производства_продукции_СХ_%     42
Доля_городского_населения_%              0
Использование_свеж_воды_млн_м3           0
Индекс_промыш_производства_%            21
Забол_инф_на_1000                      352
Коэф_смерт_насел                        40
Числ_насел                              40
Оборот_вод_млн_м3                       40
dtype: int64

In [15]:
# В стобце "Забол_инф_на_1000" много пропусков, удалим его
df = df.drop('Забол_инф_на_1000', axis=1)
df.head()

,Регион,Год,V_сточ_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
0,Алтайский край,2000,31.0,176660.0,46736.8,17660.5,122.0,52.8,569.0,109.2,14.3,2641.1,1216.0
1,Алтайский край,2001,34.0,111590.0,61854.4,23509.0,104.5,53.1,599.0,109.4,14.7,2621.1,998.0
2,Алтайский край,2002,36.0,67466.0,73107.4,27991.2,103.1,53.2,563.0,100.1,15.7,2602.6,983.0
3,Алтайский край,2003,36.0,50114.0,88733.3,34295.8,100.8,53.5,516.0,105.5,15.9,2572.0,984.0
4,Алтайский край,2004,36.0,38270.0,114840.5,44934.9,99.5,53.7,488.0,102.6,15.9,2539.4,969.0


In [16]:
df.shape

(1992, 13)

#### Удаляем регион с пропусками

In [17]:
df = df[df['Регион'] != 'Чеченская Республика']

#### Меняем формат числа у ВРП в 2023 году

In [18]:
df[df['Год'] == 2023] # сейчас данные находятся в тыс., а нужно в миллионах, как у остальных

,Регион,Год,V_сточ_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
23,Алтайский край,2023,13.0,2247651.0,1.024355e+06,482474.4,93.1,58.5,425.0,107.1,14.4,2115.3,817.0
47,Амурская область,2023,60.0,2883951.0,7.938519e+05,1054056.2,94.2,68.5,115.0,97.2,14.1,750.1,1327.0
71,Архангельская область,2023,247.0,2286387.0,7.615896e+05,793259.7,101.5,78.1,500.0,98.8,14.1,998.1,817.0
95,Астраханская область,2023,85.0,191976.0,7.777183e+05,819951.6,103.0,63.9,561.0,100.6,11.8,946.4,1860.0
119,Белгородская область,2023,58.0,17411221.0,1.341409e+06,889768.6,105.0,65.3,224.0,104.7,13.4,1500.7,1575.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1895,Чукотский автономный округ,2023,3.0,800015.0,1.867094e+05,3895053.5,91.5,69.4,100.0,110.0,10.1,48.0,227.0
1919,Ямало-Ненецкий автономный округ,2023,25.0,105498709.0,5.379402e+06,10462220.5,80.7,85.2,189.0,97.1,5.5,516.0,338.0
1943,Ярославская область,2023,143.0,9917079.0,8.497699e+05,713444.2,105.1,80.8,180.0,107.0,14.9,1187.6,314.0
1967,г. Москва,2023,833.0,22791947.0,3.233900e+07,2463550.4,70.8,100.0,1331.0,119.0,8.8,13149.8,4593.0


In [19]:
df.loc[df['Год'] == 2023, 'ВРП_млн_руб'] = df.loc[df['Год'] == 2023, 'ВРП_млн_руб'] / 1

In [20]:
df['ВРП_млн_руб'] = df['ВРП_млн_руб'].round(1)

In [21]:
df[df['Год'] == 2023]

,Регион,Год,V_сточ_вод_млн_м3,Инвестиции_в_ООС_тыс_руб,ВРП_млн_руб,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
23,Алтайский край,2023,13.0,2247651.0,1024355.4,482474.4,93.1,58.5,425.0,107.1,14.4,2115.3,817.0
47,Амурская область,2023,60.0,2883951.0,793851.9,1054056.2,94.2,68.5,115.0,97.2,14.1,750.1,1327.0
71,Архангельская область,2023,247.0,2286387.0,761589.6,793259.7,101.5,78.1,500.0,98.8,14.1,998.1,817.0
95,Астраханская область,2023,85.0,191976.0,777718.3,819951.6,103.0,63.9,561.0,100.6,11.8,946.4,1860.0
119,Белгородская область,2023,58.0,17411221.0,1341408.9,889768.6,105.0,65.3,224.0,104.7,13.4,1500.7,1575.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1895,Чукотский автономный округ,2023,3.0,800015.0,186709.4,3895053.5,91.5,69.4,100.0,110.0,10.1,48.0,227.0
1919,Ямало-Ненецкий автономный округ,2023,25.0,105498709.0,5379401.8,10462220.5,80.7,85.2,189.0,97.1,5.5,516.0,338.0
1943,Ярославская область,2023,143.0,9917079.0,849769.9,713444.2,105.1,80.8,180.0,107.0,14.9,1187.6,314.0
1967,г. Москва,2023,833.0,22791947.0,32339001.6,2463550.4,70.8,100.0,1331.0,119.0,8.8,13149.8,4593.0


#### Нормализуем Инвестии_в_ООС по ВРП

In [22]:
df['Инвестиции_в_ООС_тыс_руб'] = df['Инвестиции_в_ООС_тыс_руб'] / df['ВРП_млн_руб']

In [23]:
df = df.drop(['Инвестиции_в_ООС_тыс_руб', 'ВРП_млн_руб'], axis=1)

In [24]:
df.head()

,Регион,Год,V_сточ_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2,14.3,2641.1,1216.0
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4,14.7,2621.1,998.0
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1,15.7,2602.6,983.0
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5,15.9,2572.0,984.0
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6,15.9,2539.4,969.0


### Сохранение датасета

In [25]:
df.describe()

,Год,V_сточ_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Числ_насел,Оборот_вод_млн_м3
count,1968.000000,1968.000000,1.937000e+03,1931.000000,1968.000000,1968.000000,1954.000000,1928.000000,1928.000000
mean,2011.500000,189.902536,4.223254e+05,102.279751,70.151067,691.476479,105.074539,1781.523444,1753.348802
std,6.923946,266.510499,8.568329e+05,10.808069,12.567786,941.456301,10.109419,1775.116177,2342.734038
min,2000.000000,0.000000,6.667900e+03,41.400000,26.000000,4.720000,43.200000,40.900000,0.000000
25%,2005.750000,39.715000,9.443650e+04,97.100000,63.675000,133.490000,100.700000,755.675000,214.067500
50%,2011.500000,88.805000,2.282500e+05,101.600000,70.650000,294.000000,104.400000,1194.250000,867.970000
75%,2017.250000,216.805000,4.392505e+05,106.400000,77.800000,799.000000,109.100000,2492.350000,2157.815000
max,2023.000000,2661.000000,1.199539e+07,185.000000,100.000000,6849.000000,273.700000,13149.800000,13297.000000


In [26]:
df.head()

,Регион,Год,V_сточ_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2,14.3,2641.1,1216.0
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4,14.7,2621.1,998.0
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1,15.7,2602.6,983.0
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5,15.9,2572.0,984.0
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6,15.9,2539.4,969.0


In [27]:
df.to_excel('../../data/dataset_11_11.xlsx')
df.to_csv('../../data/dataset_11_11.csv')

In [28]:
df1 = pd.read_csv('../../data/raw/dataset_11_11.csv', index_col=0)
df1.head()

,Регион,Год,V_сточ_вод_млн_м3,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Коэф_смерт_насел,Числ_насел,Оборот_вод_млн_м3
0,Алтайский край,2000,31.0,17660.5,122.0,52.8,569.0,109.2,14.3,2641.1,1216.0
1,Алтайский край,2001,34.0,23509.0,104.5,53.1,599.0,109.4,14.7,2621.1,998.0
2,Алтайский край,2002,36.0,27991.2,103.1,53.2,563.0,100.1,15.7,2602.6,983.0
3,Алтайский край,2003,36.0,34295.8,100.8,53.5,516.0,105.5,15.9,2572.0,984.0
4,Алтайский край,2004,36.0,44934.9,99.5,53.7,488.0,102.6,15.9,2539.4,969.0


In [29]:
df1.shape

(1968, 11)

In [30]:
df2 = pd.read_csv('../../data/raw/dataset_2_0_filled_linered.csv')
df2.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_норм_%,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%
0,Алтайский край,2000,31.0,0.377989,17660.5,122.0,52.8,569.0,109.2
1,Алтайский край,2001,34.0,0.180408,23509.0,104.5,53.1,599.0,109.4
2,Алтайский край,2002,36.0,0.092283,27991.2,103.1,53.2,563.0,100.1
3,Алтайский край,2003,36.0,0.056477,34295.8,100.8,53.5,516.0,105.5
4,Алтайский край,2004,36.0,0.033324,44934.9,99.5,53.7,488.0,102.6


In [31]:
df2.shape

(1968, 9)

In [32]:
df2['Числ_насел'] = df1['Числ_насел']

In [33]:
df2.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_норм_%,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Числ_насел
0,Алтайский край,2000,31.0,0.377989,17660.5,122.0,52.8,569.0,109.2,2641.1
1,Алтайский край,2001,34.0,0.180408,23509.0,104.5,53.1,599.0,109.4,2621.1
2,Алтайский край,2002,36.0,0.092283,27991.2,103.1,53.2,563.0,100.1,2602.6
3,Алтайский край,2003,36.0,0.056477,34295.8,100.8,53.5,516.0,105.5,2572.0
4,Алтайский край,2004,36.0,0.033324,44934.9,99.5,53.7,488.0,102.6,2539.4


In [34]:
df2['Инвестиции_норм_%'] = df2['Инвестиции_норм_%'].replace(0, np.nan)

In [35]:
df2.isna().sum()

Регион                                  0
Год                                     0
Объем_сточных_вод_млн_м3                0
Инвестиции_норм_%                      84
ВРП_на_душу_руб                         0
Индексы_производства_продукции_СХ_%     0
Доля_городского_населения_%             0
Использование_свеж_воды_млн_м3          0
Индекс_промыш_производства_%            0
Числ_насел                             24
dtype: int64

In [36]:
# Показать все строки, где Числ_насел содержит пропуск
df2[df2['Числ_насел'].isna()]

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_норм_%,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Числ_насел
1824,Чувашская Республика — Чувашия,2000,132.00,0.320550,17276.5,102.2,60.3,160.00,103.2,NaN
1825,Чувашская Республика — Чувашия,2001,130.00,0.499985,23253.6,103.6,60.5,157.00,107.2,NaN
1826,Чувашская Республика — Чувашия,2002,123.00,0.441780,28261.3,98.2,60.7,154.00,98.4,NaN
1827,Чувашская Республика — Чувашия,2003,126.00,0.731285,34540.2,99.2,60.9,157.00,107.8,NaN
1828,Чувашская Республика — Чувашия,2004,134.00,0.212045,45955.1,98.1,60.9,148.00,106.4,NaN
1829,Чувашская Республика — Чувашия,2005,121.00,0.762190,54001.9,100.4,61.0,131.00,107.4,NaN
1830,Чувашская Республика — Чувашия,2006,121.00,0.283506,73147.3,104.3,57.2,126.00,118.9,NaN
1831,Чувашская Республика — Чувашия,2007,118.00,0.284410,97528.7,101.1,61.4,115.00,113.2,NaN
1832,Чувашская Республика — Чувашия,2008,115.00,0.073543,122980.3,105.0,57.8,119.00,102.7,NaN
1833,Чувашская Республика — Чувашия,2009,91.00,0.321113,111300.3,104.5,58.3,107.00,74.0,NaN


In [37]:
# взял данные с википедии
years = list(range(2000, 2024))
population = [
    1334.2, 1327.7, 1313.8, 1311.7, 1305.0, 1299.3, 1292.2, 1286.2,
    1282.6, 1279.4, 1251.6, 1250.5, 1247.0, 1243.4, 1240.0, 1238.1,
    1236.6, 1235.9, 1231.1, 1223.4, 1217.8, 1186.9, 1183.9, 1173.2
]


# Создаём DataFrame с информацией по годам
df_chuvashia = pd.DataFrame({"Год": years, "Числ_насел": population})

# Заполним пропуски в исходном df2 по годам для Чувашской Республики — Чувашия
mask = (df2['Регион'] == 'Чувашская Республика — Чувашия') & (df2['Числ_насел'].isna())
df2.loc[mask, 'Числ_насел'] = df2.loc[mask, 'Год'].map(df_chuvashia.set_index('Год')['Числ_насел'])

In [38]:
# Официальные данные численности населения СПб по годам (в тысячах человек)
# Источник: Росстат, Википедия, GoGov.ru
years = list(range(2000, 2024))
population_spb = [
    4741.9,   # 2000
    4714.8,   # 2001
    4661.2,   # 2002
    4656.5,   # 2003
    4624.1,   # 2004
    4600.0,   # 2005
    4580.6,   # 2006
    4571.2,   # 2007
    4568.0,   # 2008
    4581.9,   # 2009
    4879.6,   # 2010 (переписное изменение)
    4899.3,   # 2011
    4953.2,   # 2012
    5028.0,   # 2013
    5131.9,   # 2014
    5191.7,   # 2015
    5225.7,   # 2016
    5281.6,   # 2017
    5351.9,   # 2018
    5383.9,   # 2019
    5398.1,   # 2020
    5384.3,   # 2021
    5600.0,   # 2022 (скачок из-за административных изменений)
    5600.0    # 2023
]

# Создаём DataFrame с данными по Санкт-Петербургу
df_spb = pd.DataFrame({"Год": years, "Числ_насел": population_spb})

# Заполняем пропуски в датасете для г. Санкт-Петербург
# (если твой регион называется точно так же или похоже)
mask = (df2['Регион'] == 'г. Санкт-Петербург') & (df2['Числ_насел'].isna() | (df2['Числ_насел'] > 10000))
df2.loc[mask, 'Числ_насел'] = df2.loc[mask, 'Год'].map(df_spb.set_index('Год')['Числ_насел'])

# Также заменяем явно ошибочные значения (> 10 млн)
mask_error = (df2['Регион'] == 'г. Санкт-Петербург') & (df2['Числ_насел'] > 10000)
df2.loc[mask_error, 'Числ_насел'] = df2.loc[mask_error, 'Год'].map(df_spb.set_index('Год')['Числ_насел'])

print("Данные по СПб обновлены!")


Данные по СПб обновлены!


In [39]:
# Официальные данные численности населения г. Москвы по годам (в тысячах человек)
years = list(range(2000, 2024))
population_moscow = [
    8993.9,   # 2000
    9020.7,   # 2001
    9040.3,   # 2002
    9079.6,   # 2003
    9176.5,   # 2004
    10382.8,  # 2005
    10471.0,  # 2006
    10550.7,  # 2007
    10623.2,  # 2008
    10680.8,  # 2009
    11503.5,  # 2010
    11612.9,  # 2011
    11758.7,  # 2012
    11971.9,  # 2013
    12108.3,  # 2014
    12330.1,  # 2015
    12502.5,  # 2016
    12614.1,  # 2017
    12655.1,  # 2018
    12678.0,  # 2019
    12593.1,  # 2020
    12655.5,  # 2021
    13010.1,  # 2022
    13104.2   # 2023
]

# Создаём DataFrame с данными по Москве
df_moscow = pd.DataFrame({"Год": years, "Числ_насел": population_moscow})

# Заполняем пропуски (NaN) в датасете для г. Москва по годам
mask = (df2['Регион'] == 'г. Москва')
df2.loc[mask, 'Числ_насел'] = df2.loc[mask, 'Год'].map(df_moscow.set_index('Год')['Числ_насел'])

# Проверяем результат
print(df2[df2['Регион'] == 'г. Москва'][['Год', 'Числ_насел']].head(10))
print("Данные по Москве обновлены!")

       Год  Числ_насел
1920  2000      8993.9
1921  2001      9020.7
1922  2002      9040.3
1923  2003      9079.6
1924  2004      9176.5
1925  2005     10382.8
1926  2006     10471.0
1927  2007     10550.7
1928  2008     10623.2
1929  2009     10680.8
Данные по Москве обновлены!


In [40]:
# Официальные данные численности населения Чукотского АО по годам (в тысячах человек)
# Источник: Росстат, Википедия, GoGov.ru
years = list(range(2000, 2024))
population_chukotka = [
    61.6,    # 2000
    57.5,    # 2001
    53.8,    # 2002
    53.1,    # 2003
    51.4,    # 2004
    50.7,    # 2005
    50.5,    # 2006
    50.5,    # 2007
    50.3,    # 2008
    49.5,    # 2009
    50.5,    # 2010
    50.3,    # 2011
    51.0,    # 2012
    50.8,    # 2013
    50.6,    # 2014
    50.5,    # 2015
    50.2,    # 2016
    49.8,    # 2017
    49.3,    # 2018
    49.7,    # 2019
    50.3,    # 2020
    49.5,    # 2021
    47.9,    # 2022
    47.5     # 2023
]

# Создаём DataFrame с данными по Чукотскому АО
df_chukotka = pd.DataFrame({"Год": years, "Числ_насел": population_chukotka})

# Заполняем для Чукотского автономного округа
mask = (df2['Регион'] == 'Чукотский автономный округ')
df2.loc[mask, 'Числ_насел'] = df2.loc[mask, 'Год'].map(df_chukotka.set_index('Год')['Числ_насел'])

# Проверяем результат
print(df2[df2['Регион'] == 'Чукотский автономный округ'][['Год', 'Числ_насел']].head(10))
print("Данные по Чукотскому АО обновлены!")

       Год  Числ_насел
1848  2000        61.6
1849  2001        57.5
1850  2002        53.8
1851  2003        53.1
1852  2004        51.4
1853  2005        50.7
1854  2006        50.5
1855  2007        50.5
1856  2008        50.3
1857  2009        49.5
Данные по Чукотскому АО обновлены!


In [41]:
df2.loc[df2['Год'] == 2023, 'ВРП_на_душу_руб'] = df2.loc[df2['Год'] == 2023, 'ВРП_на_душу_руб'] * 1000

In [42]:
print(df2[['Регион', 'Год', 'ВРП_на_душу_руб']].head(24))

            Регион   Год  ВРП_на_душу_руб
0   Алтайский край  2000          17660.5
1   Алтайский край  2001          23509.0
2   Алтайский край  2002          27991.2
3   Алтайский край  2003          34295.8
4   Алтайский край  2004          44934.9
5   Алтайский край  2005          53812.4
6   Алтайский край  2006          69852.0
7   Алтайский край  2007          90759.9
8   Алтайский край  2008         106019.5
9   Алтайский край  2009         109088.7
10  Алтайский край  2010         124955.8
11  Алтайский край  2011         137975.9
12  Алтайский край  2012         154560.6
13  Алтайский край  2013         175666.0
14  Алтайский край  2014         189679.2
15  Алтайский край  2015         209023.2
16  Алтайский край  2016         230047.1
17  Алтайский край  2017         238056.6
18  Алтайский край  2018         256081.7
19  Алтайский край  2019         280785.0
20  Алтайский край  2020         300706.5
21  Алтайский край  2021         400038.7
22  Алтайский край  2022         4

In [43]:
df2.isna().sum()

Регион                                  0
Год                                     0
Объем_сточных_вод_млн_м3                0
Инвестиции_норм_%                      84
ВРП_на_душу_руб                         0
Индексы_производства_продукции_СХ_%     0
Доля_городского_населения_%             0
Использование_свеж_воды_млн_м3          0
Индекс_промыш_производства_%            0
Числ_насел                              0
dtype: int64

In [44]:
df2.head()

,Регион,Год,Объем_сточных_вод_млн_м3,Инвестиции_норм_%,ВРП_на_душу_руб,Индексы_производства_продукции_СХ_%,Доля_городского_населения_%,Использование_свеж_воды_млн_м3,Индекс_промыш_производства_%,Числ_насел
0,Алтайский край,2000,31.0,0.377989,17660.5,122.0,52.8,569.0,109.2,2641.1
1,Алтайский край,2001,34.0,0.180408,23509.0,104.5,53.1,599.0,109.4,2621.1
2,Алтайский край,2002,36.0,0.092283,27991.2,103.1,53.2,563.0,100.1,2602.6
3,Алтайский край,2003,36.0,0.056477,34295.8,100.8,53.5,516.0,105.5,2572.0
4,Алтайский край,2004,36.0,0.033324,44934.9,99.5,53.7,488.0,102.6,2539.4


In [45]:
df2.to_csv('../../data/raw/dataset_24_11.csv')

In [46]:
df2.to_excel('../../data/raw/dataset_24_11.xlsx', index=False)

In [ ]:
cols = df2.columns[:4]        
df4 = df2[cols]
df4.head()

df4.to_csv("main_dataset.csv", index=False)
df4.to_excel("main_dataset.xlsx", index=False)
